# Data Cleaning Pipeline

This notebook runs the reproducible synthetic-data generator and cleaning pipeline, then inspects the data-quality report. Every generated record is fictional. Run cells from top to bottom.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "scripts").exists(), "Open this notebook from the project folder."
print(f"Project: {PROJECT_ROOT.resolve()}")

## Generate and clean

The source intentionally contains duplicates, missing descriptive values, inconsistent category casing, and a few invalid dates. Cleaning keeps high-value outliers but records them for review.

In [ ]:
subprocess.run([sys.executable, "scripts/generate_data.py"], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable, "scripts/clean_data.py"], cwd=PROJECT_ROOT, check=True)

In [ ]:
import pandas as pd

raw = pd.read_csv(PROJECT_ROOT / "data/raw_data.csv")
cleaned = pd.read_csv(PROJECT_ROOT / "data/cleaned_data.csv", parse_dates=["Order Date"])
quality = json.loads((PROJECT_ROOT / "reports/data_quality_report.json").read_text(encoding="utf-8"))
print(f"Raw rows: {len(raw):,} | Clean rows: {len(cleaned):,}")
display(pd.DataFrame([quality]).T.rename(columns={0: "Value"}))

## Validate the cleaned table

Sales and profit are recalculated from the documented formulas. The outlier report is diagnostic only; it does not silently discard high-value transactions.

In [ ]:
assert cleaned["Order Date"].notna().all()
assert cleaned["Quantity"].gt(0).all()
assert cleaned["Discount"].between(0, 0.8).all()
assert not cleaned.duplicated().any()
display(cleaned.isna().sum().to_frame("Missing values"))
display(cleaned.head())